# Wan 2.2 TI2V-5B — single-generation multi-GPU API (Kaggle T4 × 2)

This notebook is the best experimental path for your requirement: **one video-generation job uses both T4 GPUs**, rather than sending separate jobs to separate GPUs.

It uses one ComfyUI process plus **C2C Multi-GPU Split** (Wan 2.2 token/sequence parallelism) and an asynchronous FastAPI queue:
- `POST /generate` returns a job ID immediately.
- `GET /status/{job_id}` checks progress.
- `GET /result/{job_id}` downloads the MP4 when complete.

**Important limits**
- This is experimental on Kaggle T4 × 2. The C2C project documents Wan 2.1/2.2 support, but says real multi-GPU speed-up has not yet been measured on two physical GPUs.
- It is **not VRAM pooling**: each GPU holds a full model copy. It does not turn 2 × 16 GB into one 32 GB GPU.
- T4 is Turing (SM 7.5). The project's current CUDA build lists SM75 kernels, but its own GPU qualification should be treated as the deciding check on your Kaggle runtime.
- If the C2C node refuses the hardware or ComfyUI cannot load the model on both cards, this approach will not run on that session. Do not assume it is faster until benchmarked.

Set Kaggle accelerator to **GPU T4 × 2** and Internet to **On**. Keep the Cloudflare URL private: this prototype intentionally has no API key, so anyone who gets the URL can submit jobs.

## 1. Check the two GPUs and available disk

In [1]:
import os, sys, subprocess, time, json, uuid, threading, queue, requests, pathlib
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
!df -h /kaggle/working

Sun Nov 15 21:49:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install ComfyUI and the C2C distributed custom node

In [2]:
%cd /kaggle/working
if not os.path.isdir("/kaggle/working/ComfyUI"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt
%cd /kaggle/working/ComfyUI/custom_nodes
if not os.path.isdir("ComfyUI-CustomNodePacks"):
    !git clone --depth 1 https://github.com/Code2Collapse/ComfyUI-CustomNodePacks.git
%cd /kaggle/working/ComfyUI
print("ComfyUI and C2C node pack installed.")

/kaggle/working
Cloning into 'ComfyUI'...
remote: Enumerating objects: 1514, done.
remote: Counting objects: 100% (1514/1514), done.
remote: Compressing objects: 100% (1310/1310), done.
remote: Total 1514 (delta 158), reused 1122 (delta 153), pack-reused 0 (from 0)
Receiving objects: 100% (1514/1514), 12.59 MiB | 21.45 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/kaggle/working/ComfyUI
/kaggle/working/ComfyUI/custom_nodes
Cloning into 'ComfyUI-CustomNodePacks'...
remote: Enumerating objects: 832, done.
remote: Counting objects: 100% (832/832), done.
remote: Compressing objects: 100% (785/785), done.
remote: Total 832 (delta 50), reused 532 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (832/832), 6.29 MiB | 23.67 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/kaggle/working/ComfyUI
ComfyUI and C2C node pack installed.


## 3. Download Wan 2.2 TI2V-5B model files

Files are downloaded directly to their final ComfyUI folders to avoid a second Hugging Face cache copy. Existing large files are skipped.

In [3]:
COMFY = "/kaggle/working/ComfyUI"
MODEL_FILES = [
    ("https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
     "models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"),
    ("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
    ("https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors",
     "models/vae/wan2.2_vae.safetensors"),
]
def download_direct(url, rel):
    dst = os.path.join(COMFY, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 100_000_000:
        print("Already present:", rel, round(os.path.getsize(dst)/1e9, 2), "GB")
        return
    if os.path.exists(dst):
        os.remove(dst)
    print("Downloading:", rel)
    with requests.get(url, stream=True, timeout=(30, 180)) as r:
        r.raise_for_status()
        with open(dst, "wb") as f:
            for chunk in r.iter_content(8*1024*1024):
                if chunk:
                    f.write(chunk)
    print("Downloaded:", round(os.path.getsize(dst)/1e9, 2), "GB")
for url, rel in MODEL_FILES:
    download_direct(url, rel)
!df -h /kaggle/working

Downloading: models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors
Downloaded: 10.0 GB
Downloading: models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
Downloaded: 6.74 GB
Downloading: models/vae/wan2.2_vae.safetensors
Downloaded: 1.41 GB
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   17G  2.6G  88% /kaggle/working


## 4. Start one ComfyUI process that can see both GPUs

Do not set `CUDA_VISIBLE_DEVICES` to a single card. The C2C node needs to see both physical T4s.

In [4]:
%cd /kaggle/working/ComfyUI
comfy_log = open("/tmp/comfyui_multigpu.log", "w")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
     "--lowvram"],
    cwd=COMFY, stdout=comfy_log, stderr=subprocess.STDOUT
)
for attempt in range(180):
    try:
        rr = requests.get("http://127.0.0.1:8188/system_stats", timeout=3)
        if rr.ok:
            print("ComfyUI ready.")
            break
    except requests.RequestException:
        time.sleep(2)
else:
    print(open("/tmp/comfyui_multigpu.log").read()[-10000:])

/kaggle/working/ComfyUI
ComfyUI ready.


## 5. Check C2C node registration and GPU visibility

The custom node must register as `C2CSequenceParallel`. This cell checks ComfyUI's node registry and the GPU inventory endpoint indirectly via the ComfyUI object list. If the node is missing, inspect the log and restart after resolving the import error.

In [5]:
obj = requests.get("http://127.0.0.1:8188/object_info/C2CSequenceParallel", timeout=30)
print("C2C node HTTP status:", obj.status_code)
if obj.ok:
    print(json.dumps(obj.json(), indent=2)[:5000])
else:
    print(open("/tmp/comfyui_multigpu.log").read()[-10000:])
print("\\nGPU status:")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

C2C node HTTP status: 200
{
  "C2CSequenceParallel": {
    "input": {
      "required": {
        "model": [
          "MODEL"
        ],
        "gpus": [
          "INT",
          {
            "default": 0,
            "min": 0,
            "max": 64,
            "tooltip": "How many GPUs to use; 0 = all that ComfyUI sees. Each GPU holds a full copy of the model."
          }
        ],
        "strategy": [
          [
            "auto",
            "ulysses",
            "ring"
          ],
          {
            "default": "auto",
            "tooltip": "auto: Ulysses as far as the head count and fast links allow, Ring for the rest (USP). ulysses: prefer head-parallel all-to-all (best on NVLink). ring: K/V blocks passed around (best across slow links)."
          }
        ],
        "min_tokens_per_gpu": [
          "INT",
          {
            "default": 1024,
            "min": 1,
            "max": 1048576,
            "tooltip": "Calls with fewer tokens per GPU run on o

## 6. Create the asynchronous single-generation API

The API sends each job to **one ComfyUI workflow** whose model chain is:

`UNETLoader → ModelSamplingSD3 → C2CSequenceParallel → KSampler`

The C2C node uses both visible GPUs, with Ring attention selected as a conservative starting point for PCIe-connected cards. It will create a full model copy on each GPU; the weights are not sharded.

In [6]:
%%writefile /kaggle/working/api_server.py
import os, time, uuid, threading, queue, requests
from io import BytesIO
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import StreamingResponse

COMFY = "http://127.0.0.1:8188"
WIDTH, HEIGHT = 832, 480       # start conservative; raise to 1280x704 after validation
FRAMES, FPS, STEPS, CFG = 17, 24, 12, 5.0
NEGATIVE = "low quality, blurry, flicker, jitter, deformed, distorted, warped geometry, text, watermark"

app = FastAPI(title="Wan 2.2 Single-Generation Multi-GPU API")
jobs = {}
job_queue = queue.Queue()
lock = threading.Lock()
worker_started = False

def make_workflow(image_name, prompt, seed):
    return {
      "37":{"class_type":"UNETLoader","inputs":{"unet_name":"wan2.2_ti2v_5B_fp16.safetensors","weight_dtype":"default"}},
      "38":{"class_type":"CLIPLoader","inputs":{"clip_name":"umt5_xxl_fp8_e4m3fn_scaled.safetensors","type":"wan","device":"default"}},
      "39":{"class_type":"VAELoader","inputs":{"vae_name":"wan2.2_vae.safetensors"}},
      "56":{"class_type":"LoadImage","inputs":{"image":image_name}},
      "55":{"class_type":"Wan22ImageToVideoLatent","inputs":{"width":WIDTH,"height":HEIGHT,"length":FRAMES,"batch_size":1,"vae":["39",0],"start_image":["56",0]}},
      "6":{"class_type":"CLIPTextEncode","inputs":{"clip":["38",0],"text":prompt}},
      "7":{"class_type":"CLIPTextEncode","inputs":{"clip":["38",0],"text":NEGATIVE}},
      "48":{"class_type":"ModelSamplingSD3","inputs":{"model":["37",0],"shift":8}},
      "49":{"class_type":"C2CSequenceParallel","inputs":{
          "model":["48",0],
          "gpus":0,
          "strategy":"ring",
          "min_tokens_per_gpu":1024,
          "fast_link_gpus":0,
          "rank_backend":"threads"
      }},
      "3":{"class_type":"KSampler","inputs":{"model":["49",0],"positive":["6",0],"negative":["7",0],"latent_image":["55",0],"seed":seed,"steps":STEPS,"cfg":CFG,"sampler_name":"uni_pc","scheduler":"simple","denoise":1.0}},
      "8":{"class_type":"VAEDecode","inputs":{"samples":["3",0],"vae":["39",0]}},
      "57":{"class_type":"CreateVideo","inputs":{"images":["8",0],"fps":FPS}},
      "58":{"class_type":"SaveVideo","inputs":{"video":["57",0],"filename_prefix":"wan_api/output","format":"auto","codec":"auto"}}
    }

def wait_for_video(prompt_id, timeout=7200):
    deadline = time.time() + timeout
    while time.time() < deadline:
        h = requests.get(f"{COMFY}/history/{prompt_id}", timeout=30)
        h.raise_for_status()
        item = h.json().get(prompt_id)
        if item:
            status = item.get("status", {})
            if status.get("status_str") == "error":
                raise RuntimeError(str(status))
            for out in item.get("outputs", {}).values():
                for key in ("videos", "gifs"):
                    if out.get(key):
                        return out[key][0]
        time.sleep(3)
    raise TimeoutError("Generation timed out")

def worker_loop():
    while True:
        job_id, image_bytes, content_type, prompt = job_queue.get()
        try:
            jobs[job_id]["status"] = "processing"
            fname = f"job_{job_id}.upload"
            up = requests.post(f"{COMFY}/upload/image",
                files={"image":(fname,image_bytes,content_type)},
                data={"type":"input","overwrite":"true"}, timeout=180)
            up.raise_for_status()
            image_name = up.json()["name"]
            seed = int.from_bytes(os.urandom(8), "big") % (2**63-1)
            resp = requests.post(f"{COMFY}/prompt",
                json={"prompt":make_workflow(image_name,prompt,seed),"client_id":str(uuid.uuid4())},
                timeout=180)
            if not resp.ok:
                raise RuntimeError("ComfyUI queue error: " + resp.text[:3000])
            pid = resp.json()["prompt_id"]
            jobs[job_id]["comfy_prompt_id"] = pid
            video_info = wait_for_video(pid)
            vr = requests.get(f"{COMFY}/view", params={
                "filename":video_info["filename"],
                "subfolder":video_info.get("subfolder",""),
                "type":video_info.get("type","output")}, timeout=300)
            vr.raise_for_status()
            jobs[job_id]["video"] = vr.content
            jobs[job_id]["status"] = "completed"
        except Exception as e:
            jobs[job_id]["status"] = "failed"
            jobs[job_id]["error"] = str(e)
        finally:
            job_queue.task_done()

@app.on_event("startup")
def startup():
    global worker_started
    if not worker_started:
        threading.Thread(target=worker_loop, daemon=True).start()
        worker_started = True

@app.get("/health")
def health():
    try:
        r = requests.get(f"{COMFY}/system_stats", timeout=5)
        r.raise_for_status()
        return {"ok":True,"comfyui":"ready","queued":job_queue.qsize()}
    except Exception as e:
        return {"ok":False,"error":str(e)}

@app.post("/generate")
async def generate(image:UploadFile=File(...), prompt:str=Form(
    "A cinematic product video. Preserve the exact design, shape, material and details from the reference image. Smooth subtle camera movement, realistic lighting."
)):
    if image.content_type not in ("image/jpeg","image/png","image/webp"):
        raise HTTPException(415,"Upload JPG, PNG or WEBP")
    raw = await image.read()
    if not raw or len(raw) > 20*1024*1024:
        raise HTTPException(400,"Image is empty or exceeds 20 MB")
    job_id = uuid.uuid4().hex
    with lock:
        jobs[job_id] = {"status":"queued","created":time.time()}
        job_queue.put((job_id,raw,image.content_type,prompt))
    return {"job_id":job_id,"status":"queued",
            "status_url":f"/status/{job_id}","result_url":f"/result/{job_id}"}

@app.get("/status/{job_id}")
def status(job_id:str):
    j = jobs.get(job_id)
    if not j: raise HTTPException(404,"Unknown job ID")
    return {k:v for k,v in j.items() if k != "video"}

@app.get("/result/{job_id}")
def result(job_id:str):
    j = jobs.get(job_id)
    if not j: raise HTTPException(404,"Unknown job ID")
    if j["status"] in ("queued","processing"):
        raise HTTPException(202,"Still generating; poll /status/{job_id}")
    if j["status"] == "failed":
        raise HTTPException(500,j.get("error","Generation failed"))
    return StreamingResponse(BytesIO(j["video"]),media_type="video/mp4",
        headers={"Content-Disposition":'attachment; filename="wan22-output.mp4"'})

Writing /kaggle/working/api_server.py


## 7. Install API packages and launch the API

In [7]:
!pip install -q fastapi uvicorn python-multipart
api_log = open("/tmp/wan_api.log", "w")
api_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/kaggle/working", stdout=api_log, stderr=subprocess.STDOUT
)
for _ in range(45):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if r.ok:
            print(r.json())
            break
    except requests.RequestException:
        time.sleep(1)
else:
    print(open("/tmp/wan_api.log").read()[-8000:])

{'ok': True, 'comfyui': 'ready', 'queued': 0}


## 8. Optional Cloudflare quick tunnel

Only FastAPI port 8000 is exposed. ComfyUI itself remains bound to localhost. This is an unauthenticated temporary tunnel, so treat its URL like a password.

In [8]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
tunnel_log = open("/tmp/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)
time.sleep(10)
print(open("/tmp/cloudflared.log").read()[-4000:])

inux, GOVersion: go1.27.4, GoArch: amd64
2026-11-15T21:55:55Z INF Settings: map[ha-connections:1 no-autoupdate:true protocol:quic url:http://127.0.0.1:8000]
2026-11-15T21:55:55Z INF Generated Connector ID: 7e4f86c0-6d72-4ef3-bdad-b50511780156
2026-11-15T21:55:55Z INF Initial protocol quic
2026-11-15T21:55:55Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
2026-11-15T21:55:55Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-11-15T21:55:55Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
2026-11-15T21:55:55Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-11-15T21:55:55Z INF Starting metrics server on 127.0.0.1:20241/metrics
2026-11-15T21:55:55Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveID(65074) CurveP256] connIndex=0 event=0 ip=198.41.192.167
2026-11-15T21:55:55Z INF Registered tunnel connection connIndex=0 connection=53f7ebda-451b-444f-a88b-d4b304fe7a0d event=0 ip=198.41.192.167 location=sea01 protocol=quic
2026-11-15T21:5

## 9. Call the API

Submit a source image. The API returns immediately, so the client does not have to hold a Cloudflare request open during video generation.

```bash
curl -X POST "https://YOUR-TUNNEL.trycloudflare.com/generate" \
  -F "image=@jewellery.jpg" \
  -F "prompt=Slow cinematic camera movement, preserve the exact jewellery design and stones"
```

Poll:

```bash
curl "https://YOUR-TUNNEL.trycloudflare.com/status/JOB_ID"
```

Download when `status` is `completed`:

```bash
curl -L "https://YOUR-TUNNEL.trycloudflare.com/result/JOB_ID" --output output.mp4
```

## 10. Validate both GPUs before increasing quality

1. Submit one short 17-frame job at 832×480 first.
2. Watch `!nvidia-smi` in another cell while sampling. Both GPUs should show activity if C2C successfully planned multi-GPU execution.
3. Inspect the C2C plan in the ComfyUI execution logs / node UI. If the plan reports one GPU, it has fallen back to single-GPU execution.
4. Only after a successful run, edit `WIDTH, HEIGHT` to `1280, 704`, `FRAMES` to `33`, and `STEPS` to `20` in `api_server.py`, then restart the API process.

**Troubleshooting**
- Missing node: inspect `/tmp/comfyui_multigpu.log`; restart ComfyUI after fixing custom-node import errors.
- OOM: keep 832×480 and 17 frames; each GPU needs a full model copy. Lowering resolution/frame count may help activation memory but does not remove the per-GPU model-copy requirement.
- C2C refuses T4 / no valid two-GPU plan: stop here; do not assume the two cards have been combined. Use the single-GPU fallback notebook instead.
- Kaggle session shutdown loses in-memory job status and video bytes. Download results promptly.

## References
- [C2C Distributed design and compatibility](https://github.com/Code2Collapse/ComfyUI-CustomNodePacks/blob/main/docs/c2c-distributed.md)
- [C2C node inputs / implementation](https://raw.githubusercontent.com/Code2Collapse/ComfyUI-CustomNodePacks/main/c2c_distributed/nodes.py)
- [ComfyUI Wan 2.2 guide](https://docs.comfy.org/tutorials/video/wan/wan2_2)

In [9]:
import re

logs = open("/tmp/cloudflared.log").read()
urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)

print(urls[-1] if urls else "URL not found yet")

https://paid-cooper-optimization-britain.trycloudflare.com


In [10]:
# Stop the existing ComfyUI process
comfy_proc.terminate()

try:
    comfy_proc.wait(timeout=15)
except:
    comfy_proc.kill()

# Restart with both T4 GPUs visible
comfy_log = open("/tmp/comfyui_multigpu.log", "w")

comfy_proc = subprocess.Popen(
    [
        sys.executable, "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--lowvram",
        "--disable-pinned-memory"
    ],
    cwd=COMFY,
    stdout=comfy_log,
    stderr=subprocess.STDOUT
)

# Wait for ComfyUI
for i in range(120):
    try:
        r = requests.get(
            "http://127.0.0.1:8188/system_stats",
            timeout=3
        )
        if r.ok:
            print("ComfyUI restarted successfully")
            break
    except:
        time.sleep(2)
else:
    print(open("/tmp/comfyui_multigpu.log").read()[-8000:])

ComfyUI restarted successfully


In [11]:
print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
).stdout)

Sun Nov 15 21:56:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P0             26W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
import requests, os, glob

BASE = "http://127.0.0.1:8188"
PROMPT_ID = "7cbeba86-9b3c-4cdb-8a99-3bf0bd9da2ab"

# Check ComfyUI execution history
history = requests.get(
    f"{BASE}/history/{PROMPT_ID}"
).json()

print("HISTORY:")
print(history)

# Find generated videos
print("\nVIDEO FILES:")
files = glob.glob(
    "/kaggle/working/ComfyUI/output/**/*.mp4",
    recursive=True
)

for f in files:
    print(f, os.path.getsize(f), "bytes")

HISTORY:
{}

VIDEO FILES:
/kaggle/working/ComfyUI/output/wan_api/output_00001_.mp4 248263 bytes


In [13]:
import shutil

if files:
    shutil.copy2(
        files[-1],
        "/kaggle/working/wan22-output.mp4"
    )
    print("Saved: /kaggle/working/wan22-output.mp4")
else:
    print("No MP4 found")

Saved: /kaggle/working/wan22-output.mp4
